## Qwen2.5-Coder-3B
Here we will attempt to do the same optimization of inference we did with the Saleforce Codegen 350M model but this time use the better Qwen2.5-Coder-3B mode.

In [1]:
!pip install optimum[onnxruntime]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 128.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 130.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 22.8 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
model

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [5]:
prompt = "def hello_world():"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

gen = model.generate(**inputs)
output = tokenizer.decode(gen[0], skip_special_tokens=True)
print(output)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


def hello_world():
    print("Hello World")

def hello_name(name):
    print("Hello " + name)

def hello_name_with_default(name="World"):
    print("Hello " + name)

def hello_name_with_default_and_args(name="World", *args):
    print("Hello " + name)
    for arg in args:
        print(arg)

def hello_name_with_default_and_kwargs(name="World", **kwargs):
    print("Hello " + name)
    for key, value in kwargs.items():
        print(key + " = " + value)

def hello_name_with_default_and_args_and_kwargs(name="World", *args, **kwargs):
    print("Hello " + name)
    for arg in args:
        print(arg)
    for key, value in kwargs.items():
        print(key + " = " + value)

def hello_name_with_default_and_args_and_kwargs_and_default(name="World", *args, **kwargs):
    print("Hello " + name)
    for arg in args:
        print(arg)
    for key, value in kwargs.items():
        print(key + " = " + value)

def hello_name_with_default_and_args_and_kwargs_and_default_and_default(name="World", *a

In [4]:
# Let's write a method for the token generation and decoding so that we can reuse it with different prompts
def generate_text(prompt, tokenizer, model):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen = model.generate(**inputs)
    output = tokenizer.decode(gen[0],
                              skip_special_tokens=True,
                              pad_token=tokenizer.eos_token_id)
    return output


In [13]:
prompt = "Create bar chart with matplotlib"
output = generate_text(prompt, tokenizer, model)
print(output)


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Create bar chart with matplotlib

I have a list of tuples, and I want to create a bar chart with matplotlib. The list is like this:

``````[(1, 2), (2, 3), (3, 4), (4, 5), (5, 6)]
``````

I want to create a bar chart with the first element of the tuple as the x-axis and the second element as the y-axis. How can I do this?

You can use the `matplotlib.pyplot.bar` function to create a bar chart. Here's an example:

```python
import matplotlib.pyplot as plt

# Your data
data = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6)]

# Separate the x and y values
x_values = [x for x, y in data]
y_values = [y for x, y in data]

# Create the bar chart
plt.bar(x_values, y_values)

# Add labels and title
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Bar Chart')

# Show the plot
plt.show()
```

This will create a bar chart where the x-axis represents the first element of each tuple (1, 2, 3, 4, 5) and the y-axis represents the second element of each tuple (2, 3, 4, 5, 6).

If you want to customize the 

In [4]:
from optimum.onnxruntime import ORTModelForCausalLM

# To avoid Out-Of-Memory (OOM) errors during the ONNX export process for large models,
# it is often best to perform the export on the CPU, leveraging system RAM.
# Once exported, the ONNX model can then be loaded and run on the GPU.
model_onnx = ORTModelForCausalLM.from_pretrained(model_name,
    export=True,
    provider="CPUExecutionProvider"
)

Multiple distributions found for package optimum. Picked distribution: optimum
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:207: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:235: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record th

: 

: 

In [ ]:
dummy_input = tokenizer("hello", return_tensors="pt")

torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    "/content/drive/MyDrive/onnx_models/qwen-2.5-coder-3/qwen_coder.onnx",
    opset_version=17,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
    }
)

In [ ]:
onnx_path = Path("/content/drive/MyDrive/onnx_models/qwen-2.5-coder-3")
model_onnx.save_pretrained(onnx_path)

In [1]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

dynamic_quantizer = ORTQuantizer.from_pretrained(onnx_path)
dqconfig = AutoQuantizationConfig.cuda_fp16()

model_quantized_path = dynamic_quantizer.quantize(save_dir=onnx_path,
                                                  quantization_config=dqconfig
                                                  )

KeyboardInterrupt: 

Step-by-Step Breakdown
1. ORTQuantizer.from_pretrained(model)
This initializes the quantization engine. It loads your existing ONNX model and prepares it for the transformation. It inspects the graph to identify which operations (like MatMul or Gemm) can be safely converted to integer math.

2. AutoQuantizationConfig.avx512_vnni(...)
This is a high-level "preset" configuration designed specifically for Intel CPUs (specifically those with AVX-512 and VNNI instruction sets).
- AVX-512 / VNNI: These are hardware-level accelerators in the CPU that can multiply 8-bit integers in massive batches.
- is_static=False (Dynamic): This tells the model to calculate the scaling factors for the activations on the fly during inference. It’s the "easy mode" of quantization because you don't need a calibration dataset.
- per_channel=False: This means the model uses one scaling factor for the entire weight tensor rather than a separate one for each output channel. It’s slightly less accurate but faster on some older CPU architectures.

3. dynamic_quantizer.quantize(...)
This is the execution phase. It processes the model graph, applies the config, and saves the new, shrunken .onnx file to your onnx_path.

In [ ]:
print(model_quantized_path)

/content/drive/MyDrive/onnx_models/codegen-350m-onnx


In [ ]:
import os

original_model_path = os.path.join(onnx_path, "model.onnx")
quantized_model_path = os.path.join(onnx_path, "model_quantized.onnx")

original_size = os.path.getsize(original_model_path) / (1024 * 1024) # Size in MB
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024) # Size in MB

print(f"Original model size: {original_size:.2f} MB")
print(f"Quantized model size: {quantized_size:.2f} MB")
print(f"Size reduction: {((original_size - quantized_size) / original_size) * 100:.2f}%")

Original model size: 1361.74 MB
Quantized model size: 342.19 MB
Size reduction: 74.87%


In [ ]:
# We will use the Hugging Face pipeline to define the inference that needs to be
# done so that we can apply it to the three models to assess the performance
# improvement
#
from transformers import pipeline, GenerationConfig
# The GenerationConfig object is no longer needed here, as its parameters will be passed directly
# generation_config = GenerationConfig(pad_token_id=50256,
#                                      truncation=True,
#                                      max_length=12)
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer,
                device="cpu", # Explicitly set device to CPU
                pad_token_id=50256, # Pass pad_token_id directly
                truncation=True,    # Pass truncation directly
                max_length=12       # Pass max_length directly
)

Device set to use cpu


In [ ]:
# Test the pipeline
#
result = pipe(prompt)
print(result[0]['generated_text'])

Create bar chart with matplotlib's pyplot interface
        barchart = py.figure(figsize=(6, 3))
        barcode_data = py.figure()
        py.plot(values, linew


In [ ]:
# Define some methods for collecting metrics during inference
#
from contextlib import contextmanager
from time import perf_counter
from dataclasses import dataclass

@contextmanager
def track_infer_time(time_buffer):
  start_time = perf_counter()
  yield
  end_time = perf_counter()
  time_buffer.append(end_time - start_time)

@dataclass
class BenchmarkInferenceResult:
  model_inference_time: [int]
  optimized_model_path: str


In [ ]:
from tqdm import trange
PROVIDERS = {
    ("cpu", "PyTorch CPU"),
}
results = {}
inference_runs = 200
for device, label in PROVIDERS:
  time_buffer = []

  for _ in trange(inference_runs, desc=f"Tracking inference time (PyTorch vanilla model)"):
    with track_infer_time(time_buffer):
      pipe(prompt)

  results[label] = BenchmarkInferenceResult(time_buffer, None)

Tracking inference time (PyTorch vanilla model): 100%|██████████| 200/200 [07:06<00:00,  2.13s/it]


In [ ]:
# Delete the model and clear the memory before repeating the performance evaluation
# of the onnx model.
#
import gc

del pipe
del model
gc.collect()

4305

In [ ]:
# Load the ONNX version of the model from the onnx_path
#
from optimum.onnxruntime import ORTModelForCausalLM

model_onnx = ORTModelForCausalLM.from_pretrained(onnx_path, provider="CPUExecutionProvider", file_name="model.onnx")

In [ ]:
# Define the pipeline and test to make sure it works
#
pipe = pipeline("text-generation",
                model=model_onnx,
                device="cpu", # Explicitly set device to CPU
                tokenizer=tokenizer,
                pad_token_id=50256, # Pass pad_token_id directly
                truncation=True,    # Pass truncation directly
                max_length=12       # Pass max_length directly
)

result = pipe(prompt)
print(result[0]['generated_text'])

Device set to use cpu


Create bar chart with matplotlib module

# Plotting Data
fig = plt.figure( figsize=( 6 * 2, 3.5 * 2) )
#fig = plt.figure(figsize=(10,


In [ ]:
from tqdm import trange

PROVIDERS = {
    ("CPUExecutionProvider", "ONNX CPU"),
}
inference_runs = 200
for device, label in PROVIDERS:
  time_buffer = []

  for _ in trange(inference_runs, desc=f"Tracking inference time (ONNX model)"):
    with track_infer_time(time_buffer):
      pipe(prompt)

  results[label] = BenchmarkInferenceResult(time_buffer, None)

Tracking inference time (ONNX model): 100%|██████████| 200/200 [05:05<00:00,  1.53s/it]


In [ ]:
print(results)

{'PyTorch CPU': BenchmarkInferenceResult(model_inference_time=[2.1825043910002933, 2.371705896000094, 2.1657266280008116, 2.1117550439994375, 2.092701202000171, 2.106975591000264, 2.1127226529997643, 2.170117577000383, 2.406468211000174, 2.1729288730002736, 2.110897699000816, 2.1094027829994957, 2.1193907719998606, 2.1611978880000606, 2.7719895640002505, 2.101817363000009, 2.10524485499991, 1.67466616099955, 2.096990961999836, 2.1244238359995506, 2.382627081999999, 1.2592235519996393, 2.134293083000557, 2.1138535480004066, 2.1056617670001287, 2.1028213330000654, 2.1344352309997703, 2.236671712000316, 2.2040483419996235, 2.0881821969996963, 2.0762526429998616, 2.083045039999888, 2.096356048000416, 2.148231128000589, 2.1721249590000298, 2.1500385129993447, 2.112044593999599, 2.4086751449995063, 2.111207397000726, 2.136773925999478, 2.7416910279998774, 2.477042339000036, 2.3346665939998275, 2.0951835220002977, 2.1034464989998014, 2.133972537999398, 2.1719829220000975, 2.24902385400037, 2.

In [ ]:
del pipe
del model_onnx
gc.collect()

0

In [ ]:
# Load the quantized version of the model from
print(f'Loading the model from: {model_quantized_path}')
model_quantized = ORTModelForCausalLM.from_pretrained(onnx_path, provider="CPUExecutionProvider", file_name="model_quantized.onnx")


Loading the model from: /content/drive/MyDrive/onnx_models/codegen-350m-onnx


In [ ]:
# Define the pipeline and test to make sure it works
#
pipe = pipeline("text-generation",
                model=model_quantized,
                device="cpu", # Explicitly set device to CPU
                tokenizer=tokenizer,
                pad_token_id=50256, # Pass pad_token_id directly
                truncation=True,    # Pass truncation directly
                max_length=12       # Pass max_length directly
)

result = pipe(prompt)
print(result[0]['generated_text'])

Device set to use cpu


Create bar chart with matplotlib bar chart from the previous data (i.e. Bar Chart with title "Veg Veg V")
    bar_chart = (fetch_data_BarChart(series_name, '2015


In [ ]:
PROVIDERS = {
    ("CPUExecutionProvider", "Quantized ONNX CPU"),
}
inference_runs = 200
for device, label in PROVIDERS:
  time_buffer = []

  for _ in trange(inference_runs, desc=f"Tracking inference time {label}"):
    with track_infer_time(time_buffer):
      pipe(prompt)

  results[label] = BenchmarkInferenceResult(time_buffer, None)

Tracking inference time Quantized ONNX CPU: 100%|██████████| 200/200 [03:16<00:00,  1.02it/s]


In [ ]:
import numpy as np
import plotly.express as px

def plot_results(results):
  # Compute average inference time
  time_results = {k: np.mean(v.model_inference_time) * 1e3 for k, v in results.items()}

  fig = px.bar(x=time_results.keys(), y=time_results.values(),
              title="Average inference time (ms) for each provider",
              labels={'x':'Provider', 'y':'Avg Inference time (ms)'},
              text_auto='.2s')
  fig.show()

In [ ]:
plot_results(results)

In [ ]:
# Save the results dictionary into a JSON file so that it can be reloaded later without having to rerun all the inference benchmarks
#
import json
from pathlib import Path

results_path = Path("/content/drive/MyDrive/onnx_models/results.json")

# Prepare the results for JSON serialization
serializable_results = {}
for key, benchmark_result in results.items():
    serializable_results[key] = {
        "model_inference_time": benchmark_result.model_inference_time,
        "optimized_model_path": str(benchmark_result.optimized_model_path) if benchmark_result.optimized_model_path else None
    }

with open(results_path, "w") as f:
    json.dump(serializable_results, f, indent=4)

print(f"Results saved to {results_path}")

Results saved to /content/drive/MyDrive/onnx_models/results.json


In [ ]:
del pipe
del model_quantized
gc.collect()

979

In [ ]:
# load the results from the json file
#
import json
from pathlib import Path

results_path = Path("/content/drive/MyDrive/onnx_models/results.json")

# Define the BenchmarkInferenceResult dataclass again, as it's needed for reconstruction
from dataclasses import dataclass

@dataclass
class BenchmarkInferenceResult:
  model_inference_time: list
  optimized_model_path: str

loaded_serializable_results = {}
with open(results_path, "r") as f:
    loaded_serializable_results = json.load(f)

# Reconstruct the results dictionary with BenchmarkInferenceResult objects
loaded_results = {}
for key, data in loaded_serializable_results.items():
    loaded_results[key] = BenchmarkInferenceResult(
        model_inference_time=data["model_inference_time"],
        optimized_model_path=data["optimized_model_path"]
    )

print("Results loaded successfully:")
print(loaded_results)

Results loaded successfully:
{'PyTorch CPU': BenchmarkInferenceResult(model_inference_time=[2.1825043910002933, 2.371705896000094, 2.1657266280008116, 2.1117550439994375, 2.092701202000171, 2.106975591000264, 2.1127226529997643, 2.170117577000383, 2.406468211000174, 2.1729288730002736, 2.110897699000816, 2.1094027829994957, 2.1193907719998606, 2.1611978880000606, 2.7719895640002505, 2.101817363000009, 2.10524485499991, 1.67466616099955, 2.096990961999836, 2.1244238359995506, 2.382627081999999, 1.2592235519996393, 2.134293083000557, 2.1138535480004066, 2.1056617670001287, 2.1028213330000654, 2.1344352309997703, 2.236671712000316, 2.2040483419996235, 2.0881821969996963, 2.0762526429998616, 2.083045039999888, 2.096356048000416, 2.148231128000589, 2.1721249590000298, 2.1500385129993447, 2.112044593999599, 2.4086751449995063, 2.111207397000726, 2.136773925999478, 2.7416910279998774, 2.477042339000036, 2.3346665939998275, 2.0951835220002977, 2.1034464989998014, 2.133972537999398, 2.171982922

In [ ]:
plot_results(loaded_results)

In [ ]:
time_results = {k: np.mean(v.model_inference_time) * 1e3 for k, v in loaded_results.items()}
time_results_std = {k: np.std(v.model_inference_time) * 1000 for k, v in loaded_results.items()}

In [ ]:
perf_results = {}
for k, v in loaded_results.items():
  latency_list = v.model_inference_time
  latency_50 = np.percentile(latency_list, 50) * 1e3
  latency_75 = np.percentile(latency_list, 75) * 1e3
  latency_90 = np.percentile(latency_list, 90) * 1e3
  latency_95 = np.percentile(latency_list, 95) * 1e3
  latency_99 = np.percentile(latency_list, 99) * 1e3

  average_latency = np.mean(v.model_inference_time) * 1e3
  throughput = 1 * (1000 / average_latency)

  perf_results[k] = (
        average_latency,
        latency_50,
        latency_75,
        latency_90,
        latency_95,
        latency_99,
        throughput,
    )

In [ ]:
import pandas as pd

index_labels = ['Average_latency (ms)', 'Latency_P50', 'Latency_P75',
                'Latency_P90', 'Latency_P95', 'Latency_P99', 'Throughput']
perf_df = pd.DataFrame(data=perf_results, index=index_labels)
perf_df

,PyTorch CPU,ONNX CPU,Quantized ONNX CPU
Average_latency (ms),2129.946476,1526.648568,983.414324
Latency_P50,2123.129511,1408.080623,1034.687645
Latency_P75,2172.325938,1839.126345,1052.753890
Latency_P90,2222.387421,1910.024937,1069.226776
Latency_P95,2272.468230,1932.687429,1079.612645
Latency_P99,2479.688826,1972.021206,1107.149125
Throughput,0.469495,0.655030,1.016865


In [ ]:
results = loaded_results
results_df = pd.DataFrame(columns=['Provider', 'Inference_time'])
for k, v in results.items():
  for i in range(len(v.model_inference_time)):
    results_df.loc[len(results_df.index)] = [k, v.model_inference_time[i] * 1e3]

fig = px.box(results_df, x="Provider", y="Inference_time",
             points="all",
             labels={'Provider':'Provider', 'Inference_time':'Inference durations (ms)'})
fig.show()